In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install faiss-cpu sentence-transformers langchain langchain-community

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-qp6_7wuv/unsloth_5b608c6e8b864b5181ab17440f6ca4a1
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-qp6_7wuv/unsloth_5b608c6e8b864b5181ab17440f6ca4a1
  Resolved https://github.com/unslothai/unsloth.git to commit 5819e6cc28c1f2b871a22527e3bfe9a90743772e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 136.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 61.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 139.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 52.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 126.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 150.5 MB/s  0:00:00
  Created wheel for un

In [ ]:
import json
import torch
import os
from huggingface_hub import hf_hub_download
from unsloth import FastLanguageModel
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings


print("Λήψη FAISS Database από Hugging Face...")
hf_repo = "deanwashburn/Qwen2.5-7B-ERP-Order-Extraction"

os.makedirs("erp_faiss_index", exist_ok=True)
hf_hub_download(repo_id=hf_repo, filename="erp_faiss_index 2/index.faiss", local_dir="./")
hf_hub_download(repo_id=hf_repo, filename="erp_faiss_index 2/index.pkl", local_dir="./")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vector_db = FAISS.load_local(
    folder_path="./erp_faiss_index 2",
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)


print("Φόρτωση Qwen 7B Model...")
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="deanwashburn/Qwen2.5-7B-ERP-Order-Extraction",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""


def process_order_to_json(raw_order_text):

    search_results = vector_db.similarity_search_with_score(raw_order_text, k=5)

    valid_products = []
    THRESHOLD = 8.0


    for doc, score in search_results:
        if score < THRESHOLD:
            valid_products.append(str(doc.metadata))

    if valid_products:
        print(f"[RAG Debug] Βρέθηκαν {len(valid_products)} ταυτίσεις προϊόντων στη βάση.")
        db_info = "\n".join(valid_products)
    else:
        print("[RAG Security] Αδύναμη ταύτιση προϊόντων. Κλείδωμα κωδικών.")
        db_info = "ΠΡΟΣΟΧΗ: Τα προϊόντα δεν βρέθηκαν. Για sku και barcode βάλε 'N/A' και για unit_price βάλε 0.0"


    input_text = f"Δεδομένα Αποθήκης:\n{db_info}\n\nΠαραγγελία: {raw_order_text}"


    instruction = """Εξάγαγε τα δεδομένα της παραγγελίας συμπληρώνοντας ΑΥΣΤΗΡΑ το παρακάτω JSON template.
Αντίγραψε το ΟΝΟΜΑ του πελάτη ΑΚΡΙΒΩΣ όπως αναφέρεται μέσα στο κείμενο της παραγγελίας.
Χρησιμοποίησε τα 'Δεδομένα Αποθήκης' για να βρεις το σωστό SKU, το Barcode και την Τιμή Μονάδας (unit_price) για ΚΑΘΕ αντικείμενο ξεχωριστά, κάνοντας match την περιγραφή.

Αναμενόμενο JSON Template:
{
  "customer": {
    "name": "<ΤΟ ΑΚΡΙΒΕΣ ΟΝΟΜΑ ΑΠΟ ΤΟ ΚΕΙΜΕΝΟ>"
  },
  "order_items": [
    {
      "category": "<λιπάσματα, σπόροι, ή ζωοτροφές>",
      "item_description": "<ΠΕΡΙΓΡΑΦΗ ΕΙΔΟΥΣ>",
      "sku": "<ΤΟ SKU Ή 'N/A'>",
      "barcode": "<ΤΟ BARCODE Ή 'N/A'>",
      "quantity": <ΑΡΙΘΜΟΣ>,
      "unit": "<ΜΟΝΑΔΑ>",
      "unit_price": <ΤΙΜΗ ΜΟΝΑΔΑΣ (ΑΡΙΘΜΟΣ) Ή 0.0>,
      "total_price": 0.0
    }
  ],
  "total_order_value": 0.0
}"""


    inputs = tokenizer(
        [alpaca_prompt.format(instruction, input_text, "")],
        return_tensors="pt"
    ).to("cuda")


    outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True, temperature=0.0, do_sample=False)
    full_response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]


    raw_json_str = full_response.split("### Response:\n")[-1].strip()

    if raw_json_str.startswith("```json"):
        raw_json_str = raw_json_str[7:]
    elif raw_json_str.startswith("```"):
        raw_json_str = raw_json_str[3:]

    if raw_json_str.endswith("```"):
        raw_json_str = raw_json_str[:-3]

    try:
        parsed_json = json.loads(raw_json_str.strip())


        total_value = 0.0
        for item in parsed_json.get("order_items", []):
            try:
                q = float(item.get("quantity", 0))
                up = float(item.get("unit_price", 0))
                item_total = round(q * up, 2)
                item["total_price"] = item_total
                total_value += item_total
            except (ValueError, TypeError):
                pass

        parsed_json["total_order_value"] = round(total_value, 2)
        parsed_json["delivery_notes"] = raw_order_text
        final_json = json.dumps(parsed_json, indent=2, ensure_ascii=False)
    except json.JSONDecodeError:
        final_json = raw_json_str.strip() + f'\n\n[Python Injection] "delivery_notes": "{raw_order_text}"'

    return final_json


print("\n" + "="*45)
print("=== Σύστημα Εισαγωγής Παραγγελιών AI (7B) ===")
print("Πληκτρολόγησε 'exit' ή 'έξοδος' για τερματισμό.")
print("="*45 + "\n")

while True:
    user_input = input("Γράψε την παραγγελία: ")

    if user_input.strip().lower() in ['exit', 'έξοδος', 'quit']:
        print("Τερματισμός δοκιμών.")
        break

    if not user_input.strip():
        continue

    print("\nΕπεξεργασία...")
    result = process_order_to_json(user_input)

    print("\n--- Παραγόμενο JSON ---")
    print(result)
    print("-" * 45 + "\n")

Λήψη FAISS Database από Hugging Face...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Φόρτωση Qwen 7B Model...
==((====))==  Unsloth 2026.10.11: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.10.11 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).

=== Σύστημα Εισαγωγής Παραγγελιών AI (7B) ===
Πληκτρολόγησε 'exit' ή 'έξοδος' για τερματισμό.



KeyboardInterrupt: Interrupted by user

In [ ]:
demo_orders = [
    # --- 5 Παραγγελίες με Υπαρκτούς Πελάτες (Βάση RAG) ---
    "Από τον Αγροτικό Συνεταιρισμό Θεσσαλίας, θέλουμε 100 τσουβάλια νιτρική αμμωνία και 50 κιλά σπόρο για καλαμπόκι. Στείλτε τα Λάρισα.",
    "Είμαι ο Γεώργιος Παπαδόπουλος από Καρδίτσα. Χρειάζομαι επειγόντως 2 παλέτες φύραμα για κότες και 10 σακιά λίπασμα 20-20-20.",
    "Παραγγελία για Συνεταιρισμός Αμπελουργών Νάουσας: 500 κιλά σύνθετο λίπασμα NPK και 5 μπουκάλια υγρό λίπασμα.",
    "Βάλε για Δημήτριος Δημητρίου στη Λαμία, 20 κιλά θειική αμμωνία και 10 σακουλάκια σπόροι ντομάτας.",
    "Από την Αγροτική Εταιρεία 'Η Δήμητρα' Ο.Ε., στείλτε 30 σακιά φύραμα για χοίρους και 5 τόνους σπόρο ηλίανθου.",

    # --- 5 Παραγγελίες με Άγνωστους Πελάτες (Threshold Fallback -> 'N/A') ---
    "Γεια σας, ονομάζομαι Κώστας Λάμπρου. Θέλω να μου στείλετε 50 κιλά σπόρο σιτάρι. Το συντομότερο παρακαλώ.",
    "Η Μαρία Παπαδοπούλου ζήτησε 3 τσουβάλια τροφή για κουνέλια.",
    "Σημείωσε για την Ελένη Καρρά: 100 κιλά κριθάρι για τα ζώα. Θα τα πληρώσει μετρητά.",
    "Ο Γιώργος Παππάς θέλει 5 παλέτες μικτό φύραμα. Πρόσεξε να είναι καλά κλεισμένα γιατί βρέχει.",
    "Μήτσος Καραχάλιος: Θέλω 150 κιλά καλαμπόκι σπαστό (για τις κότες)."
]

print("\n" + "="*50)
print("=== ΕΝΑΡΞΗ ΑΥΤΟΜΑΤΗΣ ΕΠΙΔΕΙΞΗΣ (10 ΣΕΝΑΡΙΑ) ===")
print("="*50 + "\n")

for i, order in enumerate(demo_orders, 1):
    print(f"[{i}/10] ΕΙΣΕΡΧΟΜΕΝΗ ΠΑΡΑΓΓΕΛΙΑ:\n{order}")
    print("-" * 50)

    result = process_order_to_json(order)

    print("ΠΑΡΑΓΟΜΕΝΟ JSON:")
    print(result)
    print("="*50 + "\n")


=== ΕΝΑΡΞΗ ΑΥΤΟΜΑΤΗΣ ΕΠΙΔΕΙΞΗΣ (10 ΣΕΝΑΡΙΑ) ===

[1/10] ΕΙΣΕΡΧΟΜΕΝΗ ΠΑΡΑΓΓΕΛΙΑ:
Από τον Αγροτικό Συνεταιρισμό Θεσσαλίας, θέλουμε 100 τσουβάλια νιτρική αμμωνία και 50 κιλά σπόρο για καλαμπόκι. Στείλτε τα Λάρισα.
--------------------------------------------------


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[RAG Debug] Βρέθηκαν 5 ταυτίσεις προϊόντων στη βάση.


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ΠΑΡΑΓΟΜΕΝΟ JSON:
{
  "customer": {
    "name": "Αγροτικός Συνεταιρισμός Θεσσαλίας"
  },
  "order_items": [
    {
      "category": "Ζωοτροφές",
      "item_description": "Καλαμποκάλευρο Ζωοτροφής 50kg",
      "sku": "FEE-204",
      "barcode": "5201234567808",
      "quantity": 100,
      "unit": "τσουβάλια",
      "unit_price": 19.2,
      "total_price": 1920.0
    },
    {
      "category": "Σπόροι",
      "item_description": "Σπόρος καλαμποκιού",
      "sku": "FEED-004",
      "barcode": "5205554443333",
      "quantity": 50,
      "unit": "κιλά",
      "unit_price": 12.8,
      "total_price": 640.0
    }
  ],
  "total_order_value": 2560.0,
  "delivery_notes": "Από τον Αγροτικό Συνεταιρισμό Θεσσαλίας, θέλουμε 100 τσουβάλια νιτρική αμμωνία και 50 κιλά σπόρο για καλαμπόκι. Στείλτε τα Λάρισα."
}

[2/10] ΕΙΣΕΡΧΟΜΕΝΗ ΠΑΡΑΓΓΕΛΙΑ:
Είμαι ο Γεώργιος Παπαδόπουλος από Καρδίτσα. Χρειάζομαι επειγόντως 2 παλέτες φύραμα για κότες και 10 σακιά λίπασμα 20-20-20.
-------------------------------------

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


... (11 outputs left out)

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ΠΑΡΑΓΟΜΕΝΟ JSON:
{
  "customer": {
    "name": "Ελένη Καρρά"
  },
  "order_items": [
    {
      "category": "ζωοτροφές",
      "item_description": "Κριθάρι για ζώα",
      "sku": "FEED-CORN",
      "barcode": "5203332221113",
      "quantity": 100,
      "unit": "κιλά",
      "unit_price": 21.0,
      "total_price": 2100.0
    }
  ],
  "total_order_value": 2100.0,
  "delivery_notes": "Σημείωσε για την Ελένη Καρρά: 100 κιλά κριθάρι για τα ζώα. Θα τα πληρώσει μετρητά."
}

[9/10] ΕΙΣΕΡΧΟΜΕΝΗ ΠΑΡΑΓΓΕΛΙΑ:
Ο Γιώργος Παππάς θέλει 5 παλέτες μικτό φύραμα. Πρόσεξε να είναι καλά κλεισμένα γιατί βρέχει.
--------------------------------------------------
[RAG Security] Αδύναμη ταύτιση προϊόντων. Κλείδωμα κωδικών.


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ΠΑΡΑΓΟΜΕΝΟ JSON:
{
  "customer": {
    "name": "Γιώργος Παππάς"
  },
  "order_items": [
    {
      "category": "ζωοτροφές",
      "item_description": "Μικτό φύραμα",
      "sku": "ZOO-FOOD-MIX",
      "barcode": "N/A",
      "quantity": 5,
      "unit": "παλέτες",
      "unit_price": 0.0,
      "total_price": 0.0
    }
  ],
  "total_order_value": 0.0,
  "delivery_notes": "Ο Γιώργος Παππάς θέλει 5 παλέτες μικτό φύραμα. Πρόσεξε να είναι καλά κλεισμένα γιατί βρέχει."
}

[10/10] ΕΙΣΕΡΧΟΜΕΝΗ ΠΑΡΑΓΓΕΛΙΑ:
Μήτσος Καραχάλιος: Θέλω 150 κιλά καλαμπόκι σπαστό (για τις κότες).
--------------------------------------------------
[RAG Debug] Βρέθηκαν 5 ταυτίσεις προϊόντων στη βάση.
ΠΑΡΑΓΟΜΕΝΟ JSON:
{
  "customer": {
    "name": "Μήτσος Καραχάλιος"
  },
  "order_items": [
    {
      "category": "Ζωοτροφές",
      "item_description": "Καλαμπόκι σπαστό για κότες",
      "sku": "ANI-004",
      "barcode": "5201234567903",
      "quantity": 150,
      "unit": "κιλά",
      "unit_price": 19.8,
      "tota